In [3]:
"""
Apply entity TYPE resolution results (entity_level_lookup from
step2_label_type_clusters.py) back onto source_type/target_type in
your main triples file, replacing 'other' with a real category from
the closed taxonomy.

Unlike entity NAME merging (which is destructive, a wrong merge loses
real distinguishing information), a type label is low-risk to apply:
it's just a category tag, it never touches entity identity or a
measured value. Combined with the closed taxonomy's enforcement checks
(every proposed_type is guaranteed to be one of the 8 valid categories,
never a parse artifact or invented category), that means it's
reasonable to apply EVERY resolved type, not just the "coherent"
majority ones, a best-guess type from a fixed, valid taxonomy is a
strict improvement over the generic 'other' catch-all either way.

Rows are still tagged with their confidence status, so anything worth
a second look (needs_manual_review = True: a disagreed cluster's
best-guess, or a singleton) is easy to filter and revisit later.

Only fills in cells that are CURRENTLY 'other', never overwrites an
already-categorized source_type/target_type.

Designed for Jupyter/Colab execution. No __main__ guard.
"""

import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
# fully_resolved_triples.xlsx lives in PART-4-Entity-Resolution, a
# SIBLING folder to PART-5-Entity-Type-Resolution (where this script
# and its notebook live). Replace the drive/folder prefix below with
# your actual full path, e.g.:
# r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PART-4-Entity-Resolution\fully_resolved_triples.xlsx"
TRIPLES_PATH = r"..\PART-4-Entity-Resolution\fully_resolved_triples.xlsx"
SOURCE_COL = "resolved_source"
TARGET_COL = "resolved_target"
SOURCE_TYPE_COL = "source_type"
TARGET_TYPE_COL = "target_type"
OTHER_LABEL = "other"

# entity_type_resolution_review.xlsx is the CLOSED-taxonomy run you
# confirmed and accepted, NOT entity_type_resolution_review2.xlsx
# (a separate model-comparison test). Lives in the same PART-5 folder
# as this script, so a bare filename works here without a full path,
# unless you'd rather be explicit, in which case use its full path too.
TYPE_RESOLUTION_PATH = "entity_type_resolution_review.xlsx"
LOOKUP_SHEET = "entity_level_lookup"

OUTPUT_TRIPLES_XLSX = "type_resolved_triples.xlsx"
OUTPUT_AUDIT_XLSX = "type_resolution_audit.xlsx"

# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)
print(f"Loaded {len(df)} triples")

lookup_df = pd.read_excel(TYPE_RESOLUTION_PATH, sheet_name=LOOKUP_SHEET)
lookup_df = lookup_df.dropna(subset=["proposed_type"])  # skip any failed classification
# case-insensitive keys: confirmed real mismatches on this corpus, e.g.
# "Sericin addition" in the triples file vs "sericin addition" in the
# lookup, same entity, different casing, a case-sensitive join would
# silently leave these as 'other' even though a valid type exists
type_lookup = dict(zip(lookup_df["entity"].astype(str).str.strip().str.lower(), lookup_df["proposed_type"]))
confidence_lookup = dict(zip(
    lookup_df["entity"].astype(str).str.strip().str.lower(),
    lookup_df["needs_manual_review"].map(lambda x: "needs_review" if x else "confident")
))
print(f"Loaded {len(type_lookup)} entity -> type mappings")

# ---------------------------------------------------------------
# APPLY: ONLY WHERE CURRENT TYPE IS 'other' AND A MAPPING EXISTS
# ---------------------------------------------------------------
def resolve_type(entity, current_type):
    if str(current_type).strip().lower() != OTHER_LABEL:
        return current_type, None  # not 'other', leave untouched
    key = str(entity).strip().lower()
    if key in type_lookup:
        return type_lookup[key], confidence_lookup[key]
    return current_type, None  # 'other' but no resolution found, leave as 'other'


new_source_types, new_target_types = [], []
source_status, target_status = [], []

for _, row in df.iterrows():
    st, st_status = resolve_type(row[SOURCE_COL], row[SOURCE_TYPE_COL])
    tt, tt_status = resolve_type(row[TARGET_COL], row[TARGET_TYPE_COL])
    new_source_types.append(st)
    new_target_types.append(tt)
    source_status.append(st_status)
    target_status.append(tt_status)

df[SOURCE_TYPE_COL] = new_source_types
df[TARGET_TYPE_COL] = new_target_types
df["source_type_resolution_status"] = source_status
df["target_type_resolution_status"] = target_status

n_source_resolved = sum(s is not None for s in source_status)
n_target_resolved = sum(s is not None for s in target_status)
print(f"\nRows with source_type resolved from 'other': {n_source_resolved}")
print(f"Rows with target_type resolved from 'other': {n_target_resolved}")

still_other_source = (df[SOURCE_TYPE_COL].astype(str).str.strip().str.lower() == OTHER_LABEL).sum()
still_other_target = (df[TARGET_TYPE_COL].astype(str).str.strip().str.lower() == OTHER_LABEL).sum()
print(f"\nStill 'other' after this pass, source: {still_other_source}, target: {still_other_target}")
print("(entities with no lookup match, e.g. new entities not covered by this run)")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
df.to_excel(OUTPUT_TRIPLES_XLSX, index=False)
print(f"\nSaved to {OUTPUT_TRIPLES_XLSX}")

readme_rows = [
    "HOW TO READ THIS FILE",
    "",
    "- type_lookup: every (entity -> proposed_type) mapping applied, with confidence status.",
    "  confident = LLM agreed a multi-member cluster or singleton was correctly classified.",
    "  needs_review = a disagreed cluster's best-guess type, or otherwise flagged, still a",
    "  valid closed-taxonomy type, just worth a second look before treating as final.",
    "",
    f"Rows resolved from 'other', source: {n_source_resolved}, target: {n_target_resolved}.",
    f"Still 'other' after this pass (no lookup match), source: {still_other_source}, "
    f"target: {still_other_target}.",
]
readme_df = pd.DataFrame({"": readme_rows})

audit_df = pd.DataFrame([
    {"entity": k, "proposed_type": v, "confidence": confidence_lookup[k]}
    for k, v in type_lookup.items()
])

with pd.ExcelWriter(OUTPUT_AUDIT_XLSX) as writer:
    readme_df.to_excel(writer, sheet_name="READ_ME_FIRST", index=False)
    audit_df.to_excel(writer, sheet_name="type_lookup", index=False)

print(f"Saved audit workbook to {OUTPUT_AUDIT_XLSX}")

Loaded 10324 triples
Loaded 23 entity -> type mappings

Rows with source_type resolved from 'other': 28
Rows with target_type resolved from 'other': 68

Still 'other' after this pass, source: 1556, target: 2579
(entities with no lookup match, e.g. new entities not covered by this run)

Saved to type_resolved_triples.xlsx
Saved audit workbook to type_resolution_audit.xlsx
